# GenAI Pipeline — Batch Input

Reads publications from `publications.db`, builds Anthropic Message Batch requests, submits
the batch, and saves metadata to `batch_jobs/` for retrieval in `genai_scope_batchoutput.ipynb`.

Batch API gives a 50% discount versus standard API pricing and runs asynchronously (results
ready within 24 hours, usually much faster).

### 1. Imports and Configuration

In [35]:
import duckdb
import pandas as pd
import json
import anthropic
import os
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

DB_PATH = "../publications.db"
BATCH_DIR = Path("batch_jobs")
RAW_SUBS_DIR = Path("raw_subsets")

In [ ]:
PROMPT_PATH = "prompt_v12b_batch.md"

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Batch API gives 50% off all input/output token prices.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $0.50         $2.50       200K
# claude-sonnet-4-6        $1.50         $7.50       1M
# claude-opus-4-8          $2.50        $12.50       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512
TEMPERATURE = 0.0


###########################################################################
# Reasoning is useful for debugging prompt quality but costs extra tokens.
# Set False for large production runs to reduce token usage.
INCLUDE_REASONING = False
# IF YOU SET AS TRUE, MAKE SURE TO USE A PROMPT WITH A REASONING INSTRUCTION OF ONE VERY CONCISE SENTENCE, OTHERWISE THE MODEL WILL GENERATE A LONG REASONING AND WASTE TOKENS.
############################################################################


client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

### 2. Data Loading

In [37]:
con = duckdb.connect(DB_PATH, read_only=True)
df = con.sql("SELECT * FROM publications_raw").df()
con.close()

print(f"Shape: {df.shape}")
print(f"\nScope distribution:\n{df['scope'].value_counts()}")
print(f"\nPillar distribution:\n{df['pillar'].value_counts()}")
df.head()

Shape: (4815, 7)

Scope distribution:
scope
out    3773
in     1042
Name: count, dtype: int64

Pillar distribution:
pillar
NA    3773
PB     695
F      151
CC      98
CM      98
Name: count, dtype: int64


,id,title,abstract,year,scope,pillar,research_category
0,pub.1192791591,Mushroom: an emerging source for next generati...,"Background: In recent years, plant-based and a...",2025,in,PB,Ingredient optimisation
1,pub.1187762379,Plant-Based Alternatives to Meat Products,Animal proteins have been used in the formulat...,2025,in,PB,Other
2,pub.1193391974,Influence of processing on protein quality and...,While meat is an established source of high-qu...,2025,in,PB,Ingredient optimisation
3,pub.1188899046,Solid State Fermentation—A Promising Approach ...,The increasing demand for sustainable dietary ...,2025,in,F,Other
4,pub.1193352421,"Physicochemical, Microbiological and Sensory E...",The bioactive properties of a phenolic extract...,2025,in,PB,End product formulation


### 3. Dataset Selection

Choose which records to send. For a full production run use the entire `df`.
Filters are commented out below as examples.

In [38]:
##########################################################################################
############################## OPTIONAL SUB-SET CREATION #################################
##########################################################################################

RANDOM_STATE = 3  # Change this to get a different random sample  - must be integer for reproducibility

def create_balanced_sample(df, n_out, n_pb, n_f, n_cultivated, n_cross, random_state=RANDOM_STATE): 
    out_sample = df[df["scope"] == "out"].sample(n=n_out, random_state=random_state)
    pb_sample = df[df["pillar"] == "PB"].sample(n=n_pb, random_state=random_state)
    f_sample = df[df["pillar"] == "F"].sample(n=n_f, random_state=random_state)
    cult_sample = df[df["pillar"] == "CM"].sample(n=n_cultivated, random_state=random_state)
    cross_sample = df[df["pillar"] == "CC"].sample(n=n_cross, random_state=random_state)
    combined = pd.concat(
        [out_sample, pb_sample, f_sample, cult_sample, cross_sample], ignore_index=True
    )
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

# Assuming ML removes 50% of out of scope files -> 1886 out, 1042 in, 2928 total.
batch_test_50outML = create_balanced_sample(df, n_out=1886, n_pb=695, n_f=151, n_cultivated=98, n_cross=98)
print(f"batch_test_50outML: {batch_test_50outML.shape}")
print(batch_test_50outML[["scope", "pillar"]].value_counts())


batch_test_50outML: (2928, 7)
scope  pillar
out    NA        1886
in     PB         695
       F          151
       CM          98
       CC          98
Name: count, dtype: int64


In [39]:

# Apply manual scope corrections from reviewed results
CORRECTIONS_PATH = Path("batch_results/msgbatch_01AdRqiFV9DQF5LT6tFXTfqZ/claude-sonnet-4-6_results.xlsx")

corrections = pd.read_excel(CORRECTIONS_PATH, usecols=["id", "corrected_scope"])
corrections = corrections[corrections["corrected_scope"].notna()]

batch_test_50outML_corr = batch_test_50outML.merge(corrections, on="id", how="left")
mask = batch_test_50outML_corr["corrected_scope"].notna()
batch_test_50outML_corr.loc[mask, "scope"] = batch_test_50outML_corr.loc[mask, "corrected_scope"]
batch_test_50outML_corr = batch_test_50outML_corr.drop(columns=["corrected_scope"])

print(f"Corrections applied: {mask.sum()}")
print(f"Original scope:   {batch_test_50outML['scope'].value_counts().to_dict()}")
print(f"Corrected scope:  {batch_test_50outML_corr['scope'].value_counts().to_dict()}")


Corrections applied: 169
Original scope:   {'out': 1886, 'in': 1042}
Corrected scope:  {'out': 1884, 'in': 1044}


In [11]:
# Save subset to Excel as required
def save_subset(df, filename, output_dir=RAW_SUBS_DIR):
    path = output_dir / filename
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} records to {path}")

#save_subset(batch_test_50outML, f"batch_test_50outML_rand{RANDOM_STATE}.xlsx")
save_subset(batch_test_50outML_corr, f"batch_test_50outML_corr_rand{RANDOM_STATE}.xlsx")

Saved 2928 records to raw_subsets\batch_test_50outML_corr_rand3.xlsx


In [40]:
DATASET = batch_test_50outML_corr  # full dataset

# To process only records not yet labelled by LLM (if you add an llm_scope column later):
# DATASET = df[df["llm_scope"].isna()].reset_index(drop=True)

# To process a specific list of IDs:
# ids = ["pub.123", "pub.456"]
# DATASET = df[df["id"].isin(ids)].reset_index(drop=True)

print(f"Records to send: {len(DATASET)}")
DATASET[["id", "scope", "pillar"]].head()

Records to send: 2928


,id,scope,pillar
0,pub.1192230359,in,PB
1,pub.1192623689,out,NA
2,pub.1188703796,out,NA
3,pub.1194035412,out,NA
4,pub.1183225808,in,PB


### 4. Load Prompt

In [41]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

system_prompt = load_prompt()
print(system_prompt)

You are an expert in alternative proteins and food science research.

Your task is to classify a research publication based on its title and abstract. First, determine whether the publication concerns research on alternative proteins intended for use in human food, applying the definitions, boundaries, and out of scope topics below. Provide a confidence score reflecting your certainty in the scope decision. Then, if in scope, assign the relevant alternative protein category label(s).

General Definition
Alternative proteins are plant-based, fermentation-derived, or cultivated substitutes for protein-rich animal-derived foods — including both novel ingredients (e.g. single-cell proteins, mycoprotein) and direct analogues of meat, seafood, dairy, and egg products. Substitution may be at the ingredient level (e.g. a plant protein replacing whey) or the product level (e.g. plant-based dairy replacing conventional dairy), and protein need not be explicitly stated as a goal. Publications are

### 5. Build Batch Requests

The Batch API uses standard message parameters — structured output is achieved via
tool use with `tool_choice` forced to the classification tool, mirroring the
Pydantic schema used in `genai_testing.ipynb`.

In [ ]:
_TOOL_PROPERTIES = {
    "scope": {
        "type": "string",
        "enum": ["in", "out"],
        "description": "Whether the publication is in scope for alternative proteins."
    },
    "confidence": {  # ================ CONFIDENCE SCALE 1-7 (1=low, 7=high) ================ #
        "type": "integer",
        "minimum": 1,
        "maximum": 7,
        "description": "Confidence score 1-7 for the scope decision."
    },
    "plant_based": {"type": "boolean"},
    "fermentation": {"type": "boolean"},
    "cultivated": {"type": "boolean"},
    "cross_cutting": {"type": "boolean"},
    "reasoning": {
        "type": "string",
        "description": "One concise sentence explaining the scope decision."
    },
}
_REQUIRED_BASE = ["scope", "confidence", "plant_based", "fermentation", "cultivated", "cross_cutting"]

def _build_tool():
    props = {k: v for k, v in _TOOL_PROPERTIES.items() if k != "reasoning" or INCLUDE_REASONING}
    required = _REQUIRED_BASE + (["reasoning"] if INCLUDE_REASONING else [])
    return {
        "name": "classify_publication",
        "description": "Record the scope and pillar classification for a research publication.",
        "input_schema": {
            "type": "object",
            "properties": props,
            "required": required,
        }
    }

CLASSIFICATION_TOOL = _build_tool()


def build_batch_request(row):
    user_message = f"Title: {row['title']}\n\nAbstract: {row['abstract']}"
    return {
        "custom_id": row["id"].replace(".", "_"),
        "params": {
            "model": MODEL,
            "max_tokens": MAX_TOKENS,
            "temperature": TEMPERATURE,
            "system": [
                {
                    "type": "text",
                    "text": system_prompt,
                    "cache_control": {"type": "ephemeral"}
                }
            ],
            "messages": [{"role": "user", "content": user_message}],
            "tools": [CLASSIFICATION_TOOL],
            "tool_choice": {"type": "tool", "name": "classify_publication"},
        }
    }

In [43]:
batch_requests = [build_batch_request(row) for _, row in DATASET.iterrows()]

print(f"Built {len(batch_requests)} batch requests.")
print(f"\nSample custom_id:  {batch_requests[0]['custom_id']}")
print(f"Sample user msg:   {batch_requests[0]['params']['messages'][0]['content'][:120]}...")

Built 2928 batch requests.

Sample custom_id:  pub_1192230359
Sample user msg:   Title: Effect of Plant Protein Ingredients at a Range of Pre-Hydration Levels on Technological Properties of Hybrid Beef...


### 6. Submit Batch and Save Metadata

Submitting sends all requests to the Anthropic Batch API. Results will be ready
within 24 hours (typically much faster). The batch ID is saved to `batch_jobs/`
so `genai_scope_batchoutput.ipynb` can retrieve results.

In [44]:
###############################################################
BATCH_NAME = "eval_opus_conf17"  # ← set a short human-readable name; 
# used as the metadata filename and batch_results subfolder
###############################################################


batch = client.messages.batches.create(requests=batch_requests)

print(f"Batch ID:  {batch.id}")
print(f"Status:    {batch.processing_status}")
print(f"Counts:    {batch.request_counts}")

BATCH_DIR.mkdir(exist_ok=True)
metadata = {
    "batch_id": batch.id,
    "batch_name": BATCH_NAME,
    "model": MODEL,
    "prompt_path": str(PROMPT_PATH),
    "n_records": len(DATASET),
    "include_reasoning": INCLUDE_REASONING,
    "created_at": datetime.now().isoformat(),
    "dataset_ids": DATASET["id"].tolist(),
}
metadata_path = BATCH_DIR / f"{BATCH_NAME}.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"\nMetadata saved to {metadata_path}")
print("\nOpen genai_scope_batchoutput.ipynb to retrieve results once the batch is complete.")

Batch ID:  msgbatch_019Ts9tiZnc3k17FnUJJbast
Status:    in_progress
Counts:    MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2928, succeeded=0)

Metadata saved to batch_jobs\eval_opus_conf17.json

Open genai_scope_batchoutput.ipynb to retrieve results once the batch is complete.
